# WurxOS Assistant v2: RAG pipeline notebook

**What this is.** WurxOS runs the day-to-day of Wurx Media, a TikTok Shop
affiliate agency (~30 staff). The v1 in-app assistant answers questions with a
regex router (`looksData` / `looksHowTo`) over a single flat knowledge table.
This project replaces it with **agentic hybrid RAG**:

- **Unstructured knowledge** (SOPs, the Knowledge Base, TikTok Shop Academy,
  app how-to guides, report narratives) is retrieved by **hybrid search**
  (vector + keyword, fused) over chunks stored in Postgres/pgvector.
- **Structured facts and numbers** (GMV, pay, attendance, incentives,
  performance) come **only from live SQL tools** — never from embeddings,
  which can't sort, sum, or guarantee an exact figure.
- The chat model is **agentic**: it decides per question whether to search
  knowledge, call a data tool, or both, instead of a brittle keyword router.

This notebook is the project's running lab notebook. Each phase gets its own
section, added below the previous one as the build progresses, so the whole
pipeline can be read and re-run end to end.

## Architecture

```mermaid
flowchart LR
    subgraph SRC["Sources"]
        R["reports · kb_articles · ai_assistant_docs · agenda_*<br/>(Postgres tables, via triggers)"]
        T["TikTok Shop Academy<br/>(weekly crawl, tts-crawl)"]
        A["App how-to guides<br/>(Assistant/app-guides/*.md)"]
    end

    R -- "trigger → ai_enqueue" --> Q[("ai_ingest_queue")]
    T -- "crawl writes tts_articles → trigger" --> Q
    A -- "sync script → ai_assistant_docs → trigger" --> Q

    Q -- "pg_cron, every minute" --> W["ai-ingest (edge function)<br/>extract → chunk → hash → embed → upsert"]

    W --> DOC[("ai_documents<br/>1 row / source item, ACL + filters")]
    W --> CH[("ai_chunks<br/>text + embedding(1536) + fts, pgvector")]

    U["User"] -- "assistant UI, SSE" --> AG["ai-chat agent loop"]
    AG -- "search_knowledge tool" --> S["ai_search()<br/>hybrid RRF, permission-checked"]
    S --> DOC
    S --> CH
    AG -- "data tools" --> LIVE[("live tables / views<br/>exact numbers, never embedded")]
    AG -- "LLM adapter" --> LLM["OpenAI (or Gemini)"]
    AG --> ANS["Cited answer<br/>[S1] knowledge · [D1] data"]
```

## Running this in Colab

1. Open this notebook in Google Colab.
2. Click the 🔑 (secrets) icon in the left sidebar and add two secrets:
   - `WURXOS_DEV_DB_URL` — the dev Postgres connection string
   - `OPEN_AI_API_KEY` — the OpenAI API key used for embeddings
3. Grant the notebook access to both secrets when prompted.
4. Run the **0. Setup** cells, then any phase section below.

The notebook also works in a plain local Jupyter: if Colab's secret manager
isn't available, it falls back to environment variables, then an interactive
`getpass` prompt — nothing is ever hard-coded here.

## Safety

**This notebook targets the DEV database only.** Every DB cell checks that the
connection string does not contain the production project ref
(`xoaaidgvblondjpvxjqp`) and refuses to run if it does. Reads default to a
read-only transaction. Any cell that writes is clearly marked
`# WARNING: WRITES TO DEV` and always rolls back. Sensitive tables (salaries,
incentives, performance, attendance, profile PII) are only ever shown as
aggregates or synthetic demo rows — never raw rows.

## 0. Setup (Colab)

Installs the few packages this notebook needs, loads the two secrets (Colab
secret manager → environment variable → interactive prompt, in that order so
this also runs in plain Jupyter), and defines `q(sql, params)`: a small helper
that runs SQL against dev over a **read-only** connection and returns a pandas
DataFrame. Every read-only cell in this notebook goes through `q()`.

In [ ]:
%pip install -q "psycopg[binary]" openai pandas matplotlib

In [ ]:
import os
import getpass

PROD_REF = "xoaaidgvblondjpvxjqp"  # WurxOS-V2 production project ref. Never touch it from here.


def _get_secret(name):
    """Colab's secret manager, then a plain env var, then an interactive prompt.

    The prompt fallback is what lets this notebook also run in a local Jupyter
    without Colab at all.
    """
    try:
        from google.colab import userdata  # only exists inside Colab
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    val = os.environ.get(name)
    if val:
        return val
    return getpass.getpass(f"{name}: ")


WURXOS_DEV_DB_URL = _get_secret("WURXOS_DEV_DB_URL")
OPEN_AI_API_KEY = _get_secret("OPEN_AI_API_KEY")

# Guard: refuse outright if this ever points at production, no matter how the
# secret got in here.
assert PROD_REF not in WURXOS_DEV_DB_URL, "Refusing: this connection string looks like PRODUCTION."
print("Secrets loaded. Connection target does not contain the production ref -- looks like dev.")

In [ ]:
import psycopg
import pandas as pd


def q(sql, params=None):
    """Run `sql` against dev, read-only, and return a pandas DataFrame.

    Every connection opens fresh and sets `default_transaction_read_only` at
    the Postgres level (not just "we promise not to write") -- belt and
    braces, since this notebook only ever needs to read for Phase 0 and most
    of Phase 1.
    """
    assert PROD_REF not in WURXOS_DEV_DB_URL, "Refusing: production ref detected."
    with psycopg.connect(WURXOS_DEV_DB_URL) as conn:
        conn.execute("SET default_transaction_read_only = on")
        with conn.cursor() as cur:
            cur.execute(sql, params)
            if cur.description is None:
                return pd.DataFrame()
            cols = [c.name for c in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)


q("select current_database() as db, current_user as user, now() as server_time")

## Phase 0: Data inventory

**Why we profiled first.** Before designing chunking, ACL modes or cost
budgets, we needed to know what the real data actually looks like on dev
(a mirror of production) — not what the original spec assumed. `phases.md`
§0.2 lists what was measured; `Assistant/inventory.md` has the full write-up.
This section reproduces the load-bearing numbers live.

**Key findings that changed the design:**

- **Four report layouts**, not one: `weekly-legacy` (289, narrative spread
  across 6 per-section `*Insights` fields), `weekly-single` (212,
  `reportInsights`), `monthly` (118, `keyWinsInsights` + `campaignsText` +
  `recommendations`), `biweekly-legacy` (5). The extractor needs a branch per
  layout, plus a generic fallback for anything that doesn't match.
- **Currency is mixed**: 488 reports have no `currency` field (defaults to USD
  by the app's own rule), 129 are explicit USD, and **7 are GBP**. Real mixed
  currency means `compare_periods` / `rank_brands` must refuse to sum or diff
  across currencies rather than silently mixing them.
- **The Knowledge Base is mostly link-only**: 148 of 150 KB articles have no
  body (Google Docs 129, Loom 12, Drive 7, other 2). The real SOP text already
  lives in `ai_assistant_docs` as **133 imported docs** (avg 1.8k chars) with a
  `<!--wurx-sop-import-->` sentinel. This surfaced an **ACL-inheritance risk**:
  25 of those KB articles are role-restricted (TL/OL/PCTL/APC) and one is
  private — if the imported SOP text were indexed as a plain, universally
  visible document, a restricted SOP would leak the moment the assistant opens
  beyond the Boss. Each imported SOP must instead be linked to its KB article
  (`ai_assistant_docs.kb_article_id`, mig 370) and inherit that article's
  visibility at query time.
- **The TikTok Shop Academy discovery**: the old seed list covered 514 essays;
  the site's own sitemap API lists **988 essays + 41 courses** (214 lessons).
  Production design (Phase 3) crawls from the sitemap instead of a hand-kept
  list, and `tts_knowledge` (0 rows in dev — not copied) is replaced going
  forward by the richer `tts_articles` staging table (mig 371).
- **Corpus size is small.** Across reports, SOPs, KB pointers and (eventually)
  TikTok content, the expected chunk count is **~8-10k**. A full backfill
  embed of everything currently indexable costs **under $0.05** at
  `text-embedding-3-small` pricing — pgvector is nowhere near a scaling
  concern at this size.

All queries below return **aggregates only** — counts, averages, distributions
— never raw narrative or PII rows, per the project's sensitive-data rule.

### Reports: by type × status

In [ ]:
df_status = q("""
    select type, status, count(*) as n
    from reports
    group by type, status
    order by type, status
""")
df_status

### Report layouts (drives the extractor branches)

In [ ]:
df_layout = q("""
    select
      type,
      case
        when type = 'monthly' then 'monthly'
        when (data ->> 'insightsSingle') = 'true' then 'weekly-single'
        else type || '-legacy'
      end as layout,
      count(*) as n
    from reports
    group by 1, 2
    order by n desc
""")
df_layout

### Currency distribution (why cross-currency math must be refused)

In [ ]:
df_currency = q("""
    select coalesce(nullif(data ->> 'currency', ''), 'USD') as currency, count(*) as n
    from reports
    group by 1
    order by n desc
""")
df_currency

### Narrative fields: how often filled, and how long (HTML stripped)

Report text is stored as HTML-ish strings inside `reports.data`. `regexp_replace`
strips tags in SQL so the length numbers reflect actual prose, not markup.

In [ ]:
NARRATIVE_FIELDS = [
    "reportInsights", "operationalUpdates", "recommendations",
    "upcomingCampaigns", "keyWinsInsights", "campaignsText",
]

rows = []
for field in NARRATIVE_FIELDS:
    df = q(
        """
        select
          count(*) filter (where btrim(coalesce(data ->> %(f)s, '')) not in ('', 'N/A')) as filled,
          avg(length(regexp_replace(coalesce(data ->> %(f)s, ''), '<[^>]+>', '', 'g')))
            filter (where btrim(coalesce(data ->> %(f)s, '')) not in ('', 'N/A')) as avg_len_chars
        from reports
        """,
        {"f": field},
    )
    df.insert(0, "field", field)
    rows.append(df)

df_narrative = pd.concat(rows, ignore_index=True)
df_narrative["avg_len_chars"] = df_narrative["avg_len_chars"].astype(float).round(0)
df_narrative

### Knowledge Base: visibility modes and the link-only vs body split

In [ ]:
df_kb = q("""
    select
      visibility,
      count(*) as n,
      count(*) filter (where btrim(coalesce(body, '')) = '') as link_only,
      count(*) filter (where btrim(coalesce(body, '')) <> '') as has_body
    from kb_articles
    group by 1
    order by n desc
""")
df_kb

### A couple of charts

Report volume by status (are drafts really excluded from what we'd index?)
and the narrative-field fill rate, side by side.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

df_status.groupby("status")["n"].sum().sort_values().plot(
    kind="barh", ax=axes[0], color="#4C72B0"
)
axes[0].set_title("Reports by status")
axes[0].set_xlabel("count")
axes[0].set_ylabel("")

df_narrative.set_index("field")["filled"].sort_values().plot(
    kind="barh", ax=axes[1], color="#55A868"
)
axes[1].set_title("Narrative fields: rows filled")
axes[1].set_xlabel("count")
axes[1].set_ylabel("")

fig.tight_layout()
plt.show()

## Phase 1: Knowledge store & permission-aware search

**Goal:** one store for everything retrievable, with a search function that
returns only what the caller is allowed to see, and the existing TikTok
Academy corpus path folded into it.

### Store design: documents vs. chunks, and why

| Table | Holds | Why separate from chunks |
|---|---|---|
| `ai_documents` | One row per **source item**: title, URL, ACL keys (`acl_mode`, `acl_roles`, `acl_users`), `brand_id`/period filters, a document-level hash | The permission check and the "did anything change?" check both happen **once per document**, not once per chunk. Deleting a document cascades to its chunks automatically. |
| `ai_chunks` | The retrievable passages: `content` (what the model reads), `embed_text` (exactly what was embedded), `embedding vector(1536)`, a generated `fts` column, a per-chunk hash | Small chunks keep retrieval precise; a per-chunk hash means editing one insight in a report re-embeds **one** chunk, not the whole document. |

No RLS `select` policy exists on either table for `authenticated` — they're
`revoke`d entirely (mig 367). The **only** way to read them is
`public.ai_search()`, a single enforcement point instead of one policy to keep
in sync per table.

### ACL modes (`ai_documents.acl_mode`)

Checked live by `ai_can_read_document()` (mig 368, hardened in mig 372 below)
— nothing is copied into the chunk at ingest time, so a brand reassignment or
a KB visibility change takes effect on the **next query**, with no re-index.

| Mode | Who can read it | Backing check |
|---|---|---|
| `all` | Everyone (any active role) | — |
| `roles` | Any role listed in `acl_roles` | `urole = any(acl_roles)` |
| `report` | Whoever can view that report's brand | `can_view_report(brand_id, author_id, uid)` |
| `brand` | Whoever can view that brand | `can_view_brand(owner_id, brand_id, uid)` |
| `kb` | Mirrors `kb_select` (mig 134), **fails closed**: OL/developer, the article's author, or its `submitted_by`, always; `office` visibility open to everyone; `role`/`users` visibility to the matching list; anything else — missing, unknown, or `private` — **denied** | mig 372 (see below) |
| `boss` | Boss always; `acl_roles` names any extra roles let in (e.g. Boss + OL) | `urole = any(acl_roles)` |

The Boss can always read everything, and an unknown `acl_mode` is **denied by
default** for everyone else.

### Mig 372: three Phase 1 hardening fixes (QA, 2026-09-28)

Phase 1 shipped, QA exercised it against real dev data and the ingest queue
under simulated crashes, and found three gaps worth closing **before** Phase 2
started writing to these tables. All three are one migration, 372:

- **Fail-closed KB visibility (M2).** The original `kb` branch read
  `coalesce(d.metadata->>'visibility', 'office')` — a **missing** visibility
  value silently behaved like `'office'` (visible to everyone). Mig 372
  changes the default to `coalesce(d.metadata->>'visibility', '')`, so a
  missing, unknown, or `'private'` value now matches **no** visibility branch
  and is denied to anyone but the owner, exactly mirroring `kb_select` (mig
  134). `submitted_by` was also added alongside `created_by` as an owner
  check, since either can be the article's author of record.
- **Keyword search matches ANY query word (D1).** The keyword arm used
  `websearch_to_tsquery`, which ANDs plain words together — a natural-language
  question rarely has *every* one of its words in the same short chunk, so
  real questions were quietly starving the keyword arm. Mig 372 switches to
  `plainto_tsquery` with every `&` turned into `|`, so **one** matching word
  is enough to surface a chunk; `ts_rank_cd` still ranks chunks that match
  *more* of the query's words higher. Demonstrated below.
- **Stuck queue items fail after 5 attempts, not forever (M1).** `ai_ingest_claim`
  reclaimed any `processing` row stuck for 10 minutes unconditionally — a
  worker that reliably crashes on one poisoned item would retry it forever,
  invisible in the health panel and burning an embedding call every time. It
  now parks a row as `failed` once it has been claimed 5 times and is still
  stuck, the same cap `ai_ingest_fail` already used for ordinary errors, so a
  crash counts as an attempt like any other failure.

### Hybrid search, simply

A single ranking method misses things the other catches: pure vector search
blurs exact names (brand names, SKUs, creator handles); pure keyword search
misses paraphrases ("how do I get paid" vs "commission payout process").
`ai_search()` runs both and fuses them:

1. **Semantic arm** — top 60 chunks by cosine distance (HNSW index).
2. **Keyword arm** — top 60 chunks by `ts_rank_cd`, matching a chunk if it
   contains **any** word of the query (mig 372, D1 above) — not all of them —
   while still ranking chunks that match more of the query's words higher.
3. **Reciprocal Rank Fusion (RRF)** merges the two ranked lists without
   needing to normalise cosine similarity against a text-rank score, which
   live on different, incomparable scales:

$$\text{score}(doc) = \sum_{\text{list} \in \{\text{semantic}, \text{keyword}\}} \frac{1}{k + \text{rank}_{\text{list}}(doc)}$$

   with $k = 60$. A document ranked highly by *either* arm scores well; one
   that both arms agree on scores best of all.

### Why reads go only through `ai_search`

`ai_documents`/`ai_chunks` have RLS **enabled** but carry **no policies** for
`authenticated` — direct `select` is denied outright, even for the Boss (see
the ACL test below). `ai_search` is `SECURITY DEFINER`: it resolves the
caller's `auth.uid()` (or, for the Slack relay running as `service_role`, an
explicitly passed `p_as_user`) and role **once per query**, then filters every
candidate row through `ai_can_read_document()` before ranking. One function,
one set of tests, no risk of a table gaining a client policy later and quietly
bypassing the ACL.

### The new tables, as Postgres actually sees them

In [ ]:
df_cols = q("""
    select table_name, column_name, data_type
    from information_schema.columns
    where table_schema = 'public' and table_name in ('ai_documents', 'ai_chunks')
    order by table_name, ordinal_position
""")
df_cols

In [ ]:
df_idx = q("""
    select tablename, indexname, indexdef
    from pg_indexes
    where schemaname = 'public' and tablename in ('ai_documents', 'ai_chunks')
    order by tablename, indexname
""")
for _, r in df_idx.iterrows():
    print(f"{r.tablename:14s} {r.indexname}\n    {r.indexdef}\n")

### RRF explained with two tiny ranked lists

No database needed for this part — just the formula above, applied by hand to
two short lists to see why a document that both arms like floats to the top.

In [ ]:
def rrf(rankings, k=60):
    """Reciprocal Rank Fusion: sum(1 / (k + rank)) for each doc, across lists."""
    scores = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)


semantic_top = ["report_bentgo_w12", "sop_commission_payout", "report_acme_w12", "guide_incentives_page"]
keyword_top = ["sop_commission_payout", "guide_incentives_page", "report_bentgo_w12", "tts_seller_fees"]

for doc, score in rrf([semantic_top, keyword_top]):
    print(f"{doc:26s} {score:.5f}")

# sop_commission_payout and report_bentgo_w12 are strong in BOTH lists and
# come out on top, even though report_bentgo_w12 ranked #1 semantically but
# only #3 on keywords -- exactly the "either arm, but especially both" effect.

### Keyword arm: ANY word matches, ranked by how many (mig 372, D1)

Before mig 372, the keyword arm effectively required **every** word of the
query to appear in a chunk (`websearch_to_tsquery` ANDs plain words). After
mig 372, `plainto_tsquery`'s `&` is rewritten to `|`, so **any one** query
word is enough to match — real Postgres full-text matching, reproduced here on
a tiny synthetic corpus by intersecting word sets instead of running `tsquery`
locally.

In [ ]:
import re

STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "how",
    "in", "into", "is", "it", "of", "on", "or", "that", "the", "to", "was",
    "were", "will", "with", "what", "which", "who",
}


def lexemes(text):
    """A crude stand-in for Postgres's 'english' text-search lexemes."""
    return [w for w in re.findall(r"[a-z0-9']+", text.lower()) if w not in STOPWORDS]


# A tiny synthetic corpus of chunk texts -- no real client data.
CORPUS = {
    "sop_commission_payout": "How the monthly commission payout is calculated and paid to each APC.",
    "guide_incentives_page": "The Incentives page shows your commission tier and payout status.",
    "report_bentgo_w12": "Bentgo's GMV Max campaign drove most of this week's commission.",
    "tts_seller_fees": "TikTok Shop seller fees are separate from affiliate commission payouts.",
    "guide_attendance": "Mark attendance for the day from the Attendance page.",
}

query = "commission payout"
q_words = set(lexemes(query))
print("query lexemes:", q_words)

rows = []
for doc, text in CORPUS.items():
    doc_words = set(lexemes(text))
    matched = q_words & doc_words
    rows.append(
        {
            "document": doc,
            "matched_words": ", ".join(sorted(matched)) or "(none)",
            "AND (pre-372) match?": matched == q_words and bool(matched),
            "OR (mig 372) rank score": len(matched),  # stands in for ts_rank_cd: more matches ranks higher
        }
    )

df_keyword_demo = pd.DataFrame(rows).sort_values("OR (mig 372) rank score", ascending=False)
df_keyword_demo

**What to look at:** `sop_commission_payout` and `guide_incentives_page` both
match every query word and score highest. Under the pre-372 AND rule,
`report_bentgo_w12` and `tts_seller_fees` (one matching word each -- this
crude demo has no stemming, so `tts_seller_fees`'s "payout**s**" doesn't
count as a match for "payout" the way Postgres's real `'english'` dictionary
would) would have been invisible to the keyword arm entirely, even though
they're clearly relevant -- they only had a chance through the semantic arm's
RRF contribution. `guide_attendance` matches nothing and is excluded either
way.

### Permission demo: the ACL matrix, live

This reproduces the shape of the hard-gate test in
`Assistant/tests/sql/acl.sql` (QA's script, run via
`node Assistant/scripts/db-dev.mjs sql ...`) directly in the notebook: create a
few synthetic documents covering different `acl_mode`s, search as three real
dev users of different roles, and show who sees what.

**This cell writes to dev** (temporary rows), unlike every other cell above.
It uses its own connection — not the read-only `q()` helper — and **always
rolls back**, so nothing persists no matter how the cell exits.

In [ ]:
# WARNING: WRITES TO DEV (rolled back) -- uses its own, non-read-only connection.
import json

DEMO_SETUP_SQL = """
select set_config('t.boss', (
  select id::text from profiles where role = 'boss' and is_active and deleted_at is null limit 1
), true);
select set_config('t.apc', (
  select p.id::text from profiles p
   where p.role = 'apc' and p.is_active and p.deleted_at is null
     and exists (select 1 from brands b where public.can_view_brand(b.owner_id, b.id, p.id))
   limit 1
), true);
select set_config('t.brand', (
  select b.id::text from brands b
   where public.can_view_brand(b.owner_id, b.id, current_setting('t.apc')::uuid)
   limit 1
), true);
select set_config('t.tl', (
  -- a TL who specifically can NOT view t.brand, so the matrix below is
  -- deterministic instead of accidentally true for an unrelated reason
  select p.id::text from profiles p
   where p.role = 'tl' and p.is_active and p.deleted_at is null
     and not public.can_view_brand(
       (select owner_id from brands where id = current_setting('t.brand')::uuid),
       current_setting('t.brand')::uuid, p.id)
   limit 1
), true);

insert into ai_documents (source_type, source_id, title, acl_mode, acl_roles, acl_users, brand_id, author_id, metadata, doc_hash)
values
  ('notebook_demo', 'nb1', 'Open to everyone',    'all',    '{}',   '{}', null, null, '{}', 'x'),
  ('notebook_demo', 'nb2', 'Boss + OL only',       'boss',   '{ol}', '{}', null, null, '{}', 'x'),
  ('notebook_demo', 'nb3', 'This brand''s report', 'report', '{}',   '{}', current_setting('t.brand')::uuid, current_setting('t.boss')::uuid, '{}', 'x'),
  ('notebook_demo', 'nb4', 'TL-only SOP',          'kb',     '{tl}', '{}', null, current_setting('t.boss')::uuid, '{"visibility": "role"}', 'x');

insert into ai_chunks (document_id, chunk_index, content, embed_text, token_count, content_hash, embedding_model)
select id, 0, title || ' zqxnotebookdemo', title || ' zqxnotebookdemo', 5, 'x', 'none'
  from ai_documents where source_type = 'notebook_demo';
"""

DOC_TITLES = ["Open to everyone", "Boss + OL only", "This brand's report", "TL-only SOP"]

conn = psycopg.connect(WURXOS_DEV_DB_URL)  # NOT read-only: this demo needs to insert + set role
rows = []
try:
    with conn.cursor() as cur:
        cur.execute(DEMO_SETUP_SQL)
        for setting, role_label in [("t.boss", "boss"), ("t.apc", "apc"), ("t.tl", "tl")]:
            cur.execute(f"select current_setting('{setting}')")
            uid = cur.fetchone()[0]
            cur.execute(
                "select set_config('request.jwt.claims', %s, true)",
                (json.dumps({"sub": uid, "role": "authenticated"}),),
            )
            cur.execute("set local role authenticated")
            cur.execute(
                "select title from ai_search(null, 'zqxnotebookdemo', array['notebook_demo'])"
            )
            seen = {r[0] for r in cur.fetchall()}
            cur.execute("reset role")
            for title in DOC_TITLES:
                rows.append({"role": role_label, "document": title, "visible": title in seen})
finally:
    conn.rollback()   # always -- nothing from this demo is meant to persist
    conn.close()

matrix = pd.DataFrame(rows).pivot(index="document", columns="role", values="visible")
matrix = matrix[["boss", "apc", "tl"]]
matrix

**What to look at:** the Boss row should be all `True`. `This brand's
report` should be `True` only for the `apc` we deliberately picked because
they can view that brand. `TL-only SOP` should be `True` only for `tl`. If any
of those come out wrong, that's the same failure the ACL hard gate in
`Assistant/tests/sql/acl.sql` is built to catch before Phase 2 ever starts
writing to these tables.

### Embed one query

A single embedding call, just to prove the pipeline end to end and see the
shape of what `ai_search` actually compares against. `text-embedding-3-small`
returns 1536 numbers per input; a handful of query strings like this costs a
fraction of a cent.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPEN_AI_API_KEY)

resp = client.embeddings.create(
    model="text-embedding-3-small",
    input="How do I submit a weekly report for a brand?",
)
vec = resp.data[0].embedding

print(f"vector length: {len(vec)}")
print(f"first 5 values: {vec[:5]}")

## Phase 2: The ingestion engine

**Goal:** every change to an indexed source shows up in the store
automatically within minutes, with retries, debounce, and a full backfill --
covering the Phase 1 sources (KB, curated docs, reports, TikTok articles)
plus incentive plans. Phase 1 built the store and the read side
(`ai_search`); Phase 2 is everything that keeps it fed: the shared library
(`supabase/functions/_shared/ai/`), the `ai-ingest` worker, the triggers, and
the write-path SQL in migrations 373, 374, 375 and 376.

### The pipeline

```mermaid
flowchart LR
    SRC["reports · kb_articles<br/>ai_assistant_docs · tts_articles<br/>incentives"] -- "INSERT/UPDATE/DELETE trigger" --> ENQ["ai_enqueue()<br/>debounce + priority + op merge"]
    ENQ --> Q[("ai_ingest_queue<br/>pending / processing / done / failed")]
    CRON["pg_cron, every minute<br/>(skipped when the queue is empty)"] --> W
    Q -- "ai_ingest_claim()<br/>SKIP LOCKED, 5-attempt cap" --> W["ai-ingest worker"]
    W --> LOAD["load the LIVE row<br/>ai_ingest_source()"]
    LOAD -- "not indexable" --> DEL["ai_ingest_delete()<br/>soft or hard"]
    LOAD -- "indexable" --> EXT["extractor.toDocument()<br/>header + sections"]
    EXT --> FP{"doc_hash =<br/>sha256(doc + PIPELINE_VERSION + model)"}
    FP -- "unchanged" --> SKIP["skip -- no chunk, no API call"]
    FP -- "fp moved only" --> TOUCH["touch -- record the new fp, nothing else"]
    FP -- "changed" --> CHUNK["chunker.ts<br/>structure-aware, per-source sizes"]
    CHUNK --> CH{"content_hash<br/>per chunk"}
    CH -- "hash already in this doc" --> REUSE["reuse the stored vector"]
    CH -- "new hash" --> EMB["embed in OpenAI batches<br/>(only the new text)"]
    REUSE --> WRITE["ai_ingest_write()<br/>one transaction"]
    EMB --> WRITE
    WRITE --> DOCS[("ai_documents")]
    WRITE --> CHUNKS[("ai_chunks")]
```

Every step after "load the live row" acts on **that row as it is right now**,
never on the queued intent -- a queued `delete` for a report that got
re-submitted and re-approved before the worker got to it still re-checks
`indexable()` on the live row and upserts, because the op sitting in the
queue is only a hint (`phases.md` P2-6).

### Sources and their rules

| Source | `source_type` | Indexable when | Notes |
|---|---|---|---|
| Reports | `report` | `status = 'approved'` **only** (Owner rule, 2026-09-28, overriding the original submitted/verified/approved design) | A report that leaves `approved` (reopened, rejected) is **soft-deleted** at once -- vectors kept, hidden from search -- so re-approving the same text costs no embedding. |
| Knowledge Base | `kb_article` | `approval_status='approved'` **and** the latest `version` in its `sop_group_id` | One document per SOP group. Its linked import docs (`ai_assistant_docs.kb_article_id`) are merged in as extra sections of the **same** document and inherit the article's own ACL (`kb`) -- a restricted SOP's imported text ends up exactly as restricted as the article that owns it, never a separately-visible `all` document. |
| Curated docs | `ai_doc` | `ai_assistant_docs.kind='doc'`, `is_active`, not linked to a KB article, no `<!--wurx-sop-import-->` sentinel | Plain reference docs written directly in the Train tab -- SOP imports never surface as their own document, only through their KB article above. |
| TikTok Shop Academy | `tts_article` | `is_active` | Sitemap crawl of the public Academy: **1,094 unique ids** out of 1,201 crawled records (repeats merged across essay/course pages), of which **1,070 actually carry extractable text** (24 are category/index pages with nothing to embed). The crawl file exists (`data/tts-academy/articles.jsonl`); the import (`scripts/import-tts.mjs`) has not been run -- see the ingestion hold below. |
| Incentives | `incentive` | at least one incentive or bonus goal | **Plans only.** `achievedValue`, `completed*`, and a commission tier's computed `amount` are stripped in both TypeScript (`extractors/incentive.ts`) and SQL (`ai_incentive_items`, belt and braces) -- they're read-time overlays, and indexing them would make an APC's own progress update look like a content change. **Basic salary is never indexed at all.** ACL (`incentive`) mirrors the app's own `inc_select` policy exactly: the employee, Boss, any active OL, or the employee's manager. **Provisional:** confirmed only at the ingestion go/no-go the Owner has not yet given (`architecture.md` §4.4) -- switching it off is one `delete` of this source type's documents, nothing else depends on it. |

Never indexed, by construction (`ai_ingest_write`'s defence-in-depth check,
independent of what any extractor might try to send): any `source_type`
starting with `salar`, `compensation`, `attendance`, `leave`, or
`performance`.

### The Owner's rule: "only new data is ever processed"

Three mechanisms, at three different levels, make a repeated backfill or a
nightly reconcile on an already-caught-up corpus into a true no-op:

1. **Document fingerprint** (`ai_documents.doc_hash`) -- sha256 of the whole
   extracted document (header, sections, ACL, metadata) + `PIPELINE_VERSION`
   + the embedding model. Unchanged → the worker skips the document outright:
   no chunking, no API call, no write. `PIPELINE_VERSION` (`config.ts`, must
   equal `ai_pipeline_version()` in SQL -- the worker refuses to run if they
   disagree) is bumped whenever the extractor, the chunker, or a text
   normalisation rule changes, so a rule change reprocesses every affected
   document exactly **once**, after which the pipeline is idempotent again.
2. **Chunk reuse by content hash** (`ai_chunks.content_hash`) -- sha256 of
   one chunk's `embed_text` + the model. `ai_ingest_write` reuses a chunk's
   stored vector whenever its `content_hash` already exists **anywhere** in
   the document, not only at the same position -- so a paragraph that simply
   moved because an earlier section grew is never re-embedded.
3. **Volatile fields never enter `embed_text`.** A report's `status`, an
   incentive's `verified` / `payout_cleared`, a TikTok article's view count --
   these live in `ai_documents.metadata` only, never inside a section's
   `text`. A metadata-only change still rewrites the document row (one cheap
   SQL statement), but touches zero chunks and calls OpenAI zero times.

A second backfill on an already-caught-up corpus enqueues nothing:
`ai_backfill_candidates()` (mig 373) compares exactly these fingerprints
against what SQL can already see, without running a single extractor.

### Debounce, soft delete, and the ingestion hold

- **Debounce.** `ai_enqueue()` pushes a pending item's `not_before` forward on
  every re-trigger instead of processing each change -- a report re-saved 10
  times in 2 minutes by autosave enqueues **one** upsert, claimable 180 s
  after the *last* save. KB content changes debounce 30 s; deletes and ACL
  changes (mig 372/373: an ACL change skips the debounce too, since
  *tightening* visibility while a debounced upsert waits would leave the old
  ACL searchable for the whole debounce window) are due **immediately** --
  removing access must never wait on a timer.
- **Soft delete.** A source row that still exists but stopped being
  indexable (a report reopened, a KB article superseded, a curated doc
  deactivated) gets `ai_documents.status='deleted'`: hidden from `ai_search`
  at once, but its chunks and vectors are kept. Re-publishing the exact same
  text later costs **zero** embedding calls. Only a row that is truly gone
  (hard-deleted at the source) removes the document -- and cascades its
  chunks -- outright.
- **The ingestion hold.** Per `TEAM.md` rule 8 (Owner, 2026-09-28), nobody
  writes to `ai_documents` / `ai_chunks` / `tts_articles` (outside a
  rolled-back test), runs a backfill, enables the ingest cron or the
  triggers on a live table, imports the TikTok crawl, or makes a bulk
  embedding call until the Owner has set the conditions and explicitly
  approved it. `scripts/ingest.mjs --backfill` / `--drain` and
  `scripts/import-tts.mjs --apply` all refuse outright without an explicit
  `--owner-approved` flag on the command line; `--dry-run` (the default for
  `ingest.mjs`) stays read-only and needs no flag at all. This notebook
  obeys the same hold: every write below runs inside a transaction that
  always rolls back, never a real backfill.

### (a) Chunking demo: header + section chunking, re-implemented in Python

`chunker.ts` is structure-aware and recursive (A§6.1): a `metrics` section is
always one chunk; a `table` section becomes balanced groups of
`TABLE_ROWS_PER_CHUNK` rows; a `prose` section under its size limit is one
chunk, otherwise it's split by headings, then paragraphs, then sentences, and
packed greedily up to `target` tokens with an `overlap` tail carried into the
next chunk. Every chunk is prefixed with the document header plus
`§ <section title>` (A§6.3) -- the single biggest retrieval-quality lever for
short chunks.

This is a **small, self-contained Python re-implementation** of that
algorithm (not a call into the real TypeScript), run against a **synthetic**
weekly report -- invented brand, invented numbers, no client data -- using
the real `report` chunk parameters from `config.ts`
(`target=450, max=700, overlap=60, min=40`, tokens estimated the same way as
`text.ts`: `ceil(len(s) / 4)`).

In [ ]:
import math
import re

REPORT_PARAMS = {"target": 450, "max": 700, "overlap": 60, "min": 40}  # config.ts CHUNK_PARAMS.report


def estimate_tokens(s):
    return math.ceil(len(s) / 4)  # text.ts estimateTokens


def split_sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


def pack_prose(text, target, max_tokens, overlap, min_tokens):
    """Greedy packing to `target` tokens, carrying an `overlap`-token tail
    forward, then merging any draft under `min_tokens` into its neighbour --
    the same three moves as chunker.ts's pack()/mergeSmall()."""
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    pieces = []
    for p in paragraphs:
        pieces.extend([p] if estimate_tokens(p) <= max_tokens else split_sentences(p))

    drafts, carry, body = [], "", ""
    for piece in pieces:
        joined = f"{body} {piece}".strip() if body else piece
        fits = estimate_tokens(f"{carry}\n{joined}" if carry else joined) <= target
        if body and not fits:
            drafts.append(f"{carry}\n{body}".strip() if carry else body)
            tail_chars = min(overlap * 4, len(body) // 2)
            carry = body[-tail_chars:].strip() if tail_chars > 0 else ""
            body = piece
        else:
            body = joined
    if body:
        drafts.append(f"{carry}\n{body}".strip() if carry else body)

    merged = []
    for d in drafts:
        if merged and estimate_tokens(d) < min_tokens and estimate_tokens(f"{merged[-1]}\n{d}") <= max_tokens:
            merged[-1] = f"{merged[-1]}\n{d}"
        else:
            merged.append(d)
    return merged


def table_chunks(text, rows_per_chunk=12):
    """Balanced row groups, same shape as chunker.ts's tableDrafts()."""
    rows = [r for r in text.split("\n") if r.strip()]
    if not rows:
        return []
    groups = math.ceil(len(rows) / rows_per_chunk)
    base, extra = divmod(len(rows), groups)
    out, at = [], 0
    for g in range(groups):
        n = base + (1 if g < extra else 0)
        out.append("\n".join(rows[at:at + n]))
        at += n
    return out


def chunk_document(header, sections, params):
    chunks = []
    for section in sections:
        text = section["text"].strip()
        if not text:
            continue
        if section["kind"] == "table":
            bodies = table_chunks(text)
        elif section["kind"] == "metrics" or estimate_tokens(text) <= params["max"]:
            bodies = [text]
        else:
            bodies = pack_prose(text, params["target"], params["max"], params["overlap"], params["min"])
        for body in bodies:
            content = f"{header}\n§ {section['title']}\n{body}"
            chunks.append({"section": section["key"], "kind": section["kind"], "tokens": estimate_tokens(content), "content": content})
    return chunks


# ---- a SYNTHETIC weekly report -- invented brand, invented numbers ----
HEADER = "[Weekly report · Northwind Kitchenware (client: Northwind Holdings Ltd) · Week 20 · 2026-05-11 → 2026-05-17 · currency: USD]"

# Long enough (~750 tokens) to exceed report MAX=700 and force a split into
# more than one chunk -- a short insights field would just be one chunk.
INSIGHT_PARAGRAPHS = [
    "GMV grew 18% week on week to $18,420.50, driven mainly by the lunchbox bundle and a strong push from the top 5 creators; the blue variant of the flagship product continues to be the best seller across every channel this brand sells through.",
    "Two new creators joined the roster this week and both hit the minimum posting requirement within 48 hours of onboarding; video watch time is up 9% compared to last week, and click-through on the lunchbox bundle videos is the highest of any SKU this quarter.",
    "GMV Max spend held steady at $500 with ROI at 3.2, comfortably inside the target band the team agreed on last month; a new creative angle for the chef knife set is being tested next week to see whether it lifts click-through further.",
    "Offsite performance was flat this week: most of the growth continues to come from in-app discovery rather than external traffic, and the operations team flagged that the sample-approval queue is close to its weekly cap and may need another reviewer.",
    "Risk watch: stock on the blue variant of the flagship product is low and may run out within the next 10 days if the current sell-through rate holds; a restock order has already been placed but is not expected to land before next week's report ships.",
    "Competitor watch: a rival brand launched a cheaper three-piece bundle this week, and early creator sentiment suggests it hasn't dented demand for the lunchbox bundle yet, though it is worth tracking closely over the next couple of reporting periods.",
    "Creator mix continues to shift toward mid-tier creators with between three and ten thousand followers, who are producing a disproportionate share of this week's sell-through relative to the samples the brand has approved for them so far.",
    "The onboarding checklist for new creators was simplified this week, cutting the average time from approval to first video from four days down to just under two, which should keep the roster growing at a healthy and sustainable pace.",
    "Customer service tickets tied to the lunchbox bundle's packaging held steady week over week, with the same handful of recurring complaints about the latch mechanism that the product team is already aware of and actively working to redesign.",
    "Overall, the brand remains on track against its quarterly targets, with GMV pacing about six percent ahead of plan and no open risks beyond the blue variant's stock position noted above, which is already being actively managed by the team.",
    "The team also began collecting structured feedback from the top twenty creators this week, aiming to fold their suggestions into next month's content briefs before the next major seasonal campaign begins in earnest across every active channel.",
    "No further escalations were raised to the OL this week, and the brand's overall shop performance score held steady, keeping it comfortably inside the healthy range the team has maintained for the last several consecutive reporting periods.",
]

SYNTHETIC_REPORT_SECTIONS = [
    {"key": "summary", "title": "Overall performance", "kind": "metrics",
     "text": "Performance this week: GMV: 18,420.50; Orders: 512; ROI: 3.1. Month to date: GMV (MTD): 61,200.00."},
    {"key": "insights:all", "title": "Insights", "kind": "prose", "text": "\n\n".join(INSIGHT_PARAGRAPHS)},
    {"key": "table:creators", "title": "Top creators", "kind": "table",
     "text": "\n".join(f"- Creator: creator_{i:02d}, Videos posted: {i}, GMV: {100 * i}.00" for i in range(1, 19))},
    {"key": "written:recommendations", "title": "Recommendations & action items", "kind": "prose",
     "text": "Increase sample approvals for mid-tier creators. Action items: send 20 samples by Friday and confirm the restock ETA with the warehouse."},
]

demo_chunks = chunk_document(HEADER, SYNTHETIC_REPORT_SECTIONS, REPORT_PARAMS)
df_chunk_demo = pd.DataFrame([
    {"chunk_index": i, "section": c["section"], "kind": c["kind"], "tokens": c["tokens"],
     "preview": c["content"].split("\n", 2)[-1][:90] + "..."}
    for i, c in enumerate(demo_chunks)
])
df_chunk_demo

**What to look at:** `summary` (a `metrics` section) stays exactly **one**
chunk regardless of size. `insights:all` is long enough to exceed `MAX=700`
tokens, so it splits into two chunks at paragraph boundaries near
`TARGET=450`, with the tail of the first chunk's last paragraph carried into
the second as overlap -- run the cell and diff the two previews to see the
carried sentence repeated. `table:creators` (18 rows) splits into two
**balanced** groups of 9 rows each, not a 12-then-6 split, matching
`tableDrafts()`'s `Math.ceil(rows/rowsPer)` group count. `recommendations` is
short enough to stay one chunk. Every `content` (not shown in the preview,
which starts after the header/section line) begins with the same document
header -- the retrieval-quality lever from §6.3.

### (b) Fingerprint demo: what actually gates an OpenAI call

Two hashes matter, at two different levels (A§9.5, `hash.ts`, `chunker.ts`):

- **`ai_documents.doc_hash`** -- sha256 over the *whole* extracted document
  (header, sections, ACL, **metadata**) + `PIPELINE_VERSION` + the embed
  model. Because `metadata` is part of this hash, a genuinely volatile field
  that lives in metadata (an incentive's `verified` flag, say) **does** change
  `doc_hash` -- the document row gets rewritten. That's one cheap SQL
  statement, not an API call.
- **`ai_chunks.content_hash`** -- sha256 over just one chunk's `embed_text` +
  the model. This is what actually gates an OpenAI call: `ai_ingest_write`
  reuses a chunk's stored vector whenever its `content_hash` already exists
  anywhere in the document, so a chunk whose text didn't change is **never**
  re-embedded even when the document row above it was rewritten. Volatile
  fields (status, verified, views, progress) live in `metadata` only, never
  inside a section's `text` (source catalog, §5.1) -- which is exactly what
  keeps them out of `embed_text`, and therefore out of `content_hash`.

The demo below reproduces `content_hash` (`sha256(embed_text + "|" + model)`,
`chunker.ts`): change a document's status -- which never touches
`embed_text` -- and the hash doesn't move; change one sentence of the actual
content and it does.

In [ ]:
import hashlib

MODEL = "text-embedding-3-small"


def content_hash(embed_text, model=MODEL):
    """sha256(embed_text + '|' + model) -- exactly chunker.ts's content_hash."""
    return hashlib.sha256(f"{embed_text}|{model}".encode()).hexdigest()


HEADER = "[Weekly report · Northwind Kitchenware (client: Northwind Holdings Ltd) · Week 20 · 2026-05-11 → 2026-05-17 · currency: USD]"
SECTION_TEXT = "GMV grew 18% week on week, driven by the lunchbox bundle. Top creator doubled output."

# `embed_text` -- what actually gets embedded. Status lives in `metadata` on
# the real ai_documents row, never here, so it plays no part in this hash.
embed_text_v1 = f"{HEADER}\n§ Insights\n{SECTION_TEXT}"
fp_v1 = content_hash(embed_text_v1)
print(f"fingerprint (doc status = submitted): {fp_v1}")

# The Owner's rule in action: status moves submitted -> approved (a
# metadata-only field on the real document row), but embed_text is
# byte-for-byte the same string, so its content_hash cannot move.
fp_v2 = content_hash(embed_text_v1)   # embed_text unchanged -- status never entered it
print(f"fingerprint (doc status = approved):  {fp_v2}  -- unchanged: {fp_v2 == fp_v1}")

# Now change one sentence of the actual content.
embed_text_v3 = embed_text_v1.replace("18%", "24%")
fp_v3 = content_hash(embed_text_v3)
print(f"fingerprint (GMV 18% -> 24%):         {fp_v3}  -- changed:   {fp_v3 != fp_v1}")

**What to look at:** the first two fingerprints are identical -- the status
transition never touched `embed_text`, so this chunk's stored vector would be
reused whatever happened to `ai_documents.metadata.status` in between. The
third fingerprint, from a one-word change to the actual sentence, differs
from both. Multiply this by a real document's chunk count and it's the whole
reason a nightly reconcile over an unchanged corpus makes zero embedding
calls.

### (c) Live corpus plan: what WOULD be indexed, right now, on dev

Read-only SQL aggregates against the same rules the extractors use
(`indexable()` / `ai_ingest_source`, mig 373/376) -- nothing is queued,
nothing is written, and nothing but counts and layout labels ever leaves the
database. This is the same shape as `node scripts/ingest.mjs --dry-run`'s
per-source table, reproduced here directly in SQL.

In [ ]:
df_plan_reports = q("""
    select
      case
        when type = 'monthly' then 'monthly'
        when (data ->> 'insightsSingle') = 'true' then 'weekly-single'
        else type || '-legacy'
      end as layout,
      count(*) as approved_reports
    from reports
    where status = 'approved'   -- mig 376: the only indexable status
    group by 1
    order by approved_reports desc
""")
print("Approved reports by layout (only these would ever become ai_documents rows):")
df_plan_reports

In [ ]:
df_plan_kb = q("""
    with latest as (
      -- Latest APPROVED version per SOP group -- same window as ai_ingest_source (mig 373/376).
      select a.id,
             row_number() over (partition by coalesce(a.sop_group_id, a.id) order by a.version desc nulls last, a.id) = 1 as is_latest
        from kb_articles a
       where a.approval_status = 'approved'
    )
    select
      exists (select 1 from ai_assistant_docs d where d.kb_article_id = k.id and d.is_active) as has_linked_sop,
      count(*) as indexable_kb_articles
    from kb_articles k
    join latest l on l.id = k.id and l.is_latest
    group by 1
    order by 1
""")
print("Indexable KB articles (approved + latest version), split by whether a linked SOP doc is merged in:")
df_plan_kb

In [ ]:
df_plan_docs = q("""
    select count(*) as indexable_curated_docs
    from ai_assistant_docs
    where kind = 'doc' and is_active and kb_article_id is null
      and position('<!--wurx-sop-import-->' in coalesce(content, '')) = 0
""")

df_plan_incentives = q("""
    select count(*) as indexable_incentive_rows
    from incentives
    where jsonb_array_length(coalesce(incentives, '[]'::jsonb)) + jsonb_array_length(coalesce(bonuses, '[]'::jsonb)) > 0
""")

summary = pd.DataFrame([
    {"source": "report", "indexable_rows": int(df_plan_reports["approved_reports"].sum())},
    {"source": "kb_article", "indexable_rows": int(df_plan_kb["indexable_kb_articles"].sum())},
    {"source": "ai_doc", "indexable_rows": int(df_plan_docs["indexable_curated_docs"].iloc[0])},
    {"source": "incentive", "indexable_rows": int(df_plan_incentives["indexable_incentive_rows"].iloc[0])},
])
print("Corpus plan summary (tts_article excluded -- the crawl file lives outside the DB until imported, see below):")
summary

**What to look at:** these are exactly the row counts a real `ai_backfill()`
would enqueue on an empty store -- compare them against the "dry-run numbers"
cost table below (e), which come from the Lead's numbered dry run and include
`tts_article` (from the crawl file, not yet in this database) and the actual
resulting **chunk** and **token** counts per source, not just row counts.

### (d) End-to-end write demo: real embeddings in, `ai_search` out, then rollback

This is the one place in Phase 2 where the notebook actually exercises the
write path (`ai_ingest_write`'s shape) end to end: insert two **synthetic**
documents with **real** OpenAI embeddings for two tiny invented texts, call
`ai_search()` as the Boss and as an APC to see the ACL difference, then roll
back. It never touches the ingestion hold (no queue row, no trigger, no
`ai_backfill`) and costs two embedding calls on ~20-word strings -- a
fraction of a cent, inside the $1/task budget (`TEAM.md` rule 5).

**This cell writes to dev.** Like the Phase 1 ACL demo above, it uses its own
non-read-only connection and **always rolls back**, whatever happens inside
the `try`.

In [ ]:
# WARNING: WRITES TO DEV (rolled back) -- 2 real OpenAI embedding calls, its
# own non-read-only connection, always rolls back.
import json

from openai import OpenAI

openai_client = OpenAI(api_key=OPEN_AI_API_KEY)

# Two synthetic documents with an invented, unique term so keyword search
# can't accidentally match anything real. One is open to everyone, one is
# Boss + OL only -- the ACL difference the demo is built to show.
DOCS_P2 = [
    {
        "source_id": "nbp2-1",
        "title": "Notebook demo: open SOP snippet",
        "acl_mode": "all",
        "acl_roles": [],
        "text": "The Zzyxquartz creator-verification protocol requires re-checking a creator's TikTok handle every 90 days.",
    },
    {
        "source_id": "nbp2-2",
        "title": "Notebook demo: OL-only escalation note",
        "acl_mode": "boss",
        "acl_roles": ["ol"],
        "text": "Zzyxquartz escalations above tier 3 go straight to the OL, not the APC.",
    },
]
QUERY_TEXT = "Zzyxquartz escalation process"


def to_pgvector(vec):
    return "[" + ",".join(repr(float(x)) for x in vec) + "]"


doc_vecs = {d["source_id"]: openai_client.embeddings.create(model="text-embedding-3-small", input=d["text"]).data[0].embedding for d in DOCS_P2}
query_vec = openai_client.embeddings.create(model="text-embedding-3-small", input=QUERY_TEXT).data[0].embedding

conn = psycopg.connect(WURXOS_DEV_DB_URL)  # NOT read-only: this demo writes, then always rolls back
rows = []
try:
    with conn.cursor() as cur:
        cur.execute("select set_config('t2.boss', (select id::text from profiles where role = 'boss' and is_active and deleted_at is null limit 1), true)")
        cur.execute("select set_config('t2.apc',  (select id::text from profiles where role = 'apc'  and is_active and deleted_at is null limit 1), true)")

        for d in DOCS_P2:
            cur.execute(
                """
                insert into ai_documents (source_type, source_id, title, acl_mode, acl_roles, acl_users,
                                           brand_id, author_id, metadata, doc_hash, pipeline_version, embed_model, source_fp)
                values ('notebook_demo_p2', %s, %s, %s, %s, '{}', null, null, '{}', %s, 'notebook-demo', 'text-embedding-3-small', %s)
                returning id
                """,
                (d["source_id"], d["title"], d["acl_mode"], d["acl_roles"], f"hash-{d['source_id']}", f"fp-{d['source_id']}"),
            )
            doc_id = cur.fetchone()[0]
            cur.execute(
                """
                insert into ai_chunks (document_id, chunk_index, section, content, embed_text, token_count,
                                       content_hash, embedding, embedding_model)
                values (%s, 0, 'demo', %s, %s, %s, %s, %s::extensions.vector, 'text-embedding-3-small')
                """,
                (doc_id, d["text"], d["text"], len(d["text"]) // 4, f"chash-{d['source_id']}", to_pgvector(doc_vecs[d["source_id"]])),
            )

        for setting, role_label in [("t2.boss", "boss"), ("t2.apc", "apc")]:
            cur.execute(f"select current_setting('{setting}')")
            uid = cur.fetchone()[0]
            cur.execute("select set_config('request.jwt.claims', %s, true)", (json.dumps({"sub": uid, "role": "authenticated"}),))
            cur.execute("set local role authenticated")
            cur.execute(
                """
                select title, round(score::numeric, 5) as score
                  from ai_search(%s::extensions.vector, %s, array['notebook_demo_p2'], null, null, null, 10, null)
                 order by score desc
                """,
                (to_pgvector(query_vec), QUERY_TEXT),
            )
            seen = {title: float(score) for title, score in cur.fetchall()}
            cur.execute("reset role")
            for d in DOCS_P2:
                rows.append({"role": role_label, "document": d["title"], "score": seen.get(d["title"])})
finally:
    conn.rollback()   # always -- nothing from this demo is meant to persist
    conn.close()

df_write_demo = pd.DataFrame(rows).pivot(index="document", columns="role", values="score")
df_write_demo

**What to look at:** the Boss column should have a real score in both rows.
The APC column should have `NaN` for "OL-only escalation note" -- `ai_can_read_document`'s
`boss` branch (`urole = any(acl_roles)`) denies an `apc` role outright, even
though the chunk is a strong semantic and keyword match for the query.
`NaN` here means the row genuinely wasn't in `ai_search`'s result set for that
role, not that it scored zero.

### (e) Cost table: a full backfill, priced

These are the Lead's numbers from a full `scripts/ingest.mjs --dry-run`
(every source, including `tts_article` planned from the crawl file with
`--tts-jsonl`) -- not recomputed here, since that run needs the TikTok crawl
file and would otherwise duplicate live queries this notebook already ran in
(c). Pricing is `text-embedding-3-small` at $0.02 / 1M tokens (`ingest.mjs`'s
`PRICE_PER_M`, verified on the day of the run).

In [ ]:
PRICE_PER_M = 0.02  # text-embedding-3-small, USD per 1M tokens (ingest.mjs)

# From the Lead's full `scripts/ingest.mjs --dry-run` (2026-09-28), not
# recomputed here -- see the markdown above for why.
DRY_RUN = [
    {"source": "report", "docs": 583, "chunks": 6060, "tokens": 960_070},
    {"source": "kb_article", "docs": 149, "chunks": 174, "tokens": 62_531},
    {"source": "ai_doc", "docs": 6, "chunks": 15, "tokens": 5_943},
    {"source": "tts_article", "docs": 1070, "chunks": 5091, "tokens": 2_341_016},
    {"source": "incentive", "docs": 119, "chunks": 216, "tokens": 25_296},
]

df_cost = pd.DataFrame(DRY_RUN)
df_cost["cost_usd"] = (df_cost["tokens"] / 1e6 * PRICE_PER_M).round(4)
total = pd.DataFrame([{
    "source": "TOTAL",
    "docs": df_cost["docs"].sum(),
    "chunks": df_cost["chunks"].sum(),
    "tokens": df_cost["tokens"].sum(),
    "cost_usd": round(df_cost["tokens"].sum() / 1e6 * PRICE_PER_M, 4),
}])

df_cost = pd.concat([df_cost, total], ignore_index=True)
print(f"A full backfill of everything indexable today would embed {int(total['tokens'].iloc[0]):,} tokens "
      f"≈ ${total['cost_usd'].iloc[0]:.3f} -- comfortably inside the $1/task budget (TEAM.md rule 5).")
df_cost